# HW2: Non-linear Classification & Ensembles

In this homework, you will implement:
1. **Decision Tree Classifier** — recursive greedy splitting with Gini impurity
2. **Bagging Classifier** — bootstrap aggregating with majority vote
3. **Random Forest Classifier** — bagging with random feature subsets
4. **Gradient Boosting Classifier** — additive model with gradient descent in function space

**Requirements:**
- All implementations must be **vectorized** using NumPy (no explicit Python loops for per-sample operations) where possible. We will test for it during quizz!

**Canvas Submission:**
- Upload only this notebook with the solutions. Do not upload any additional files!

**Grading:**
- You grade for the task == proportion of passed tests at the end

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

from trees_helpers import (
    generate_demo_binary_data,
    generate_demo_multiclass_data,
    generate_demo_complex_data,
    plot_decision_boundary_2d,
    plot_tree_depth_comparison,
    plot_ensemble_vs_single,
    plot_boosting_loss_curve,
    plot_bootstrap_demo,
)

from trees_tests import (
    check_decision_tree,
    check_bagging_classifier,
    check_random_forest,
    check_gradient_boosting,
    run_single_test,
    run_tests,
)

np.random.seed(42)
plt.rcParams["figure.figsize"] = [10, 6]
plt.rcParams["font.size"] = 12

---
## Section 2: Decision Trees

A **decision tree** classifies data by recursively splitting the feature space. At each node, it picks the feature and threshold that produce the **purest** children.

**Key concepts:**
- **Gini impurity**: measures how "mixed" a set of labels is. For a set with class proportions $p_1, p_2, \ldots, p_K$:
  $$\text{Gini}(S) = 1 - \sum_{k=1}^{K} p_k^2$$
- **Greedy splitting**: at each node, try all features and thresholds, pick the one minimizing weighted Gini of children.
- **Recursive partitioning**: split until max_depth reached, node is pure, or too few samples.

Let's build a decision tree step by step!

First, let's generate some demo data to work with.

In [ ]:
X_demo, y_demo = generate_demo_binary_data()
plt.scatter(X_demo[:, 0], X_demo[:, 1], c=y_demo, cmap="RdYlBu", edgecolors="k", s=30)
plt.xlabel("Feature 1")
plt.ylabel("Feature 2")
plt.title("Demo Binary Classification Data")
plt.grid(True, alpha=0.3)
plt.show()

### Exercise 2.1: Gini Impurity

Implement the Gini impurity function. Given an array of labels `y`, compute:

$$\text{Gini}(y) = 1 - \sum_{k} p_k^2$$

where $p_k$ is the proportion of class $k$ in `y`.

In [ ]:
def gini_impurity(y):
    """Compute Gini impurity of a label array.

    Parameters
    ----------
    y : ndarray of shape (n_samples,)

    Returns
    -------
    float : Gini impurity in [0, 1)
    """
    # YOUR CODE HERE
    raise NotImplementedError("Implement gini_impurity")

In [ ]:
try:
    assert abs(gini_impurity(np.array([0, 0, 0])) - 0.0) < 1e-10, "Pure node should have Gini = 0"
    assert abs(gini_impurity(np.array([0, 1])) - 0.5) < 1e-10, "Balanced binary should have Gini = 0.5"
    assert abs(gini_impurity(np.array([0, 1, 2])) - (1 - 3 * (1 / 3) ** 2)) < 1e-10, "3-class balanced should be ~0.667"
    assert abs(gini_impurity(np.array([0, 0, 1])) - (1 - (2 / 3) ** 2 - (1 / 3) ** 2)) < 1e-10
    print("All gini_impurity tests passed! ✓")
except Exception as e:
    print(f"⚠️ Skipped (not yet implemented or error): {e}")

### Exercise 2.2: Finding the Best Split

Implement a function that finds the best feature and threshold to split on.

For each feature, try all midpoints between consecutive sorted unique values as candidate thresholds. For each candidate, compute the weighted Gini impurity of the resulting left and right children:

$$\text{Gini}_{\text{split}} = \frac{n_L}{n} \cdot \text{Gini}(y_L) + \frac{n_R}{n} \cdot \text{Gini}(y_R)$$

Return the feature index, threshold, and best Gini score.

In [ ]:
def find_best_split(X, y):
    """Find the best feature and threshold to split on.

    Parameters
    ----------
    X : ndarray of shape (n_samples, n_features)
    y : ndarray of shape (n_samples,)

    Returns
    -------
    best_feature : int or None
        Index of the best feature to split on.
    best_threshold : float or None
        Best threshold value.
    best_gini : float
        Weighted Gini impurity of the best split.
    """
    # YOUR CODE HERE
    raise NotImplementedError("Implement find_best_split")

In [ ]:
try:
    X_simple = np.array([[1, 0], [2, 0], [3, 0], [4, 0]])
    y_simple = np.array([0, 0, 1, 1])
    feat, thresh, gini_val = find_best_split(X_simple, y_simple)
    assert feat == 0, f"Should split on feature 0, got {feat}"
    assert abs(thresh - 2.5) < 1e-10, f"Should split at 2.5, got {thresh}"
    assert abs(gini_val - 0.0) < 1e-10, f"Perfect split should have Gini = 0, got {gini_val}"
    print("All find_best_split tests passed! ✓")
except Exception as e:
    print(f"⚠️ Skipped (not yet implemented or error): {e}")

### Exercise 2.3: Splitting Data

Implement a function that splits data into left and right subsets based on a feature and threshold.

In [ ]:
def split_data(X, y, feature_idx, threshold):
    """Split data into left (<=) and right (>) subsets.

    Parameters
    ----------
    X : ndarray of shape (n_samples, n_features)
    y : ndarray of shape (n_samples,)
    feature_idx : int
    threshold : float

    Returns
    -------
    X_left, y_left, X_right, y_right : ndarrays
    """
    # YOUR CODE HERE
    raise NotImplementedError("Implement split_data")

In [ ]:
try:
    X_simple = np.array([[1, 0], [2, 0], [3, 0], [4, 0]])
    y_simple = np.array([0, 0, 1, 1])
    X_left, y_left, X_right, y_right = split_data(X_simple, y_simple, 0, 2.5)
    assert len(X_left) == 2 and len(X_right) == 2, f"Expected 2+2 split, got {len(X_left)}+{len(X_right)}"
    assert np.all(y_left == 0) and np.all(y_right == 1), "Split should perfectly separate classes"
    print("All split_data tests passed! ✓")
except Exception as e:
    print(f"⚠️ Skipped (not yet implemented or error): {e}")

### Exercise 2.4: DecisionTreeClassifier

Now combine the building blocks into a full `DecisionTreeClassifier` class.

The tree is stored as a nested dictionary:
```python
# Internal node:
{"feature": int, "threshold": float, "left": ..., "right": ...}
# Leaf node:
{"value": int}  # the predicted class
```

**Methods:**
- `fit(X, y)` — build the tree recursively
- `predict(X)` — predict class for each sample

**Stopping criteria (a leaf is created when any is true):**
- Current depth equals `max_depth`
- Node is pure (all labels the same)
- Fewer than `min_samples_split` samples
- No valid split found

In [ ]:
class DecisionTreeClassifier:
    """Decision tree classifier using Gini impurity.

    Parameters
    ----------
    max_depth : int, default=10
        Maximum depth of the tree.
    min_samples_split : int, default=2
        Minimum number of samples required to split a node.
    random_state : int or None, default=None
        Not used, included for API compatibility with ensemble methods.
    """

    def __init__(self, max_depth=10, min_samples_split=2, random_state=None):
        self.max_depth = max_depth
        self.min_samples_split = min_samples_split
        self.random_state = random_state
        self.tree_ = None

    def fit(self, X, y):
        """Build the decision tree.

        Parameters
        ----------
        X : ndarray of shape (n_samples, n_features)
        y : ndarray of shape (n_samples,)

        Returns
        -------
        self
        """
        # YOUR CODE HERE
        raise NotImplementedError("Implement fit")

    def _build_tree(self, X, y, depth):
        """Recursively build the tree.

        Use the building blocks from earlier exercises.
        Return a leaf dict {"value": class} or an internal node dict
        {"feature": ..., "threshold": ..., "left": ..., "right": ...}.
        """
        # YOUR CODE HERE
        raise NotImplementedError("Implement _build_tree")

    def _predict_sample(self, x, node):
        """Predict class for a single sample by traversing the tree."""
        # YOUR CODE HERE
        raise NotImplementedError("Implement _predict_sample")

    def predict(self, X):
        """Predict class labels for samples in X.

        Parameters
        ----------
        X : ndarray of shape (n_samples, n_features)

        Returns
        -------
        ndarray of shape (n_samples,)
        """
        # YOUR CODE HERE
        raise NotImplementedError("Implement predict")

In [ ]:
try:
    run_single_test(DecisionTreeClassifier, check_decision_tree)
except Exception as e:
    print(f"⚠️ Skipped (not yet implemented or error): {e}")

### Visualizing Decision Trees

Let's see how the decision boundary changes with different `max_depth` values.

In [ ]:
try:
    X_complex, y_complex = generate_demo_complex_data()
    plot_tree_depth_comparison(X_complex, y_complex, DecisionTreeClassifier)
except Exception as e:
    print(f"⚠️ Skipped (not yet implemented or error): {e}")

---
## Section 3: Bagging

**Bootstrap Aggregating (Bagging)** reduces variance by training multiple models on different bootstrap samples and averaging their predictions.

**Key ideas:**
- **Bootstrap sample**: draw $n$ samples with replacement from the training set (some points appear multiple times, others are left out)
- **Aggregation**: combine predictions via majority vote (classification)
- Each model sees a different version of the data → decorrelates errors

Let's build a BaggingClassifier step by step!

### Exercise 3.1: Bootstrap Sampling

Implement a function that creates a bootstrap sample (sampling with replacement) from the data.

In [ ]:
def bootstrap_sample(X, y, random_state=None):
    """Create a bootstrap sample (with replacement).

    Parameters
    ----------
    X : ndarray of shape (n_samples, n_features)
    y : ndarray of shape (n_samples,)
    random_state : int or None

    Returns
    -------
    X_boot : ndarray of shape (n_samples, n_features)
    y_boot : ndarray of shape (n_samples,)
    """
    # YOUR CODE HERE
    raise NotImplementedError("Implement bootstrap_sample")

In [ ]:
try:
    X_boot, y_boot = bootstrap_sample(X_demo, y_demo, random_state=42)
    assert X_boot.shape == X_demo.shape, f"Shape mismatch: {X_boot.shape} vs {X_demo.shape}"
    assert y_boot.shape == y_demo.shape, f"Shape mismatch: {y_boot.shape} vs {y_demo.shape}"
    unique_rows = len(
        np.unique(np.arange(len(X_demo))[np.random.RandomState(42).choice(len(X_demo), size=len(X_demo), replace=True)])
    )
    assert unique_rows < len(X_demo), "Bootstrap should have duplicates"
    print(f"Bootstrap: {unique_rows} unique out of {len(X_demo)} samples")
    print("All bootstrap_sample tests passed! ✓")
except Exception as e:
    print(f"⚠️ Skipped (not yet implemented or error): {e}")

In [ ]:
try:
    plot_bootstrap_demo(X_demo, y_demo)
except Exception as e:
    print(f"⚠️ Skipped (not yet implemented or error): {e}")

### Exercise 3.2: Majority Vote

Implement a function that takes predictions from multiple estimators and returns the majority vote for each sample.

**Input:** `predictions` is an array of shape `(n_estimators, n_samples)` where each row is one model's predictions.

**Output:** array of shape `(n_samples,)` with the most common class for each sample.

In [ ]:
def majority_vote(predictions):
    """Compute majority vote from multiple estimator predictions.

    Parameters
    ----------
    predictions : ndarray of shape (n_estimators, n_samples)

    Returns
    -------
    ndarray of shape (n_samples,) : majority class for each sample
    """
    # YOUR CODE HERE
    raise NotImplementedError("Implement majority_vote")

In [ ]:
try:
    preds = np.array([[0, 1, 0], [0, 1, 1], [1, 1, 0]])
    votes = majority_vote(preds)
    assert np.array_equal(votes, np.array([0, 1, 0])), f"Expected [0, 1, 0], got {votes}"
    print("All majority_vote tests passed! ✓")
except Exception as e:
    print(f"⚠️ Skipped (not yet implemented or error): {e}")

### Exercise 3.3: BaggingClassifier

Combine bootstrap sampling and majority vote into a full `BaggingClassifier`.

**Methods:**
- `fit(X, y)` — train `n_estimators` models, each on a bootstrap sample. Pass `random_state=i` to each base estimator to ensure different random states. Forward any `base_estimator_kwargs` to the constructor.
- `predict(X)` — collect predictions from all estimators and return the majority vote

**Important:** Store the fitted estimators in `self.estimators_` (a list).

In [ ]:
class BaggingClassifier:
    """Bagging (Bootstrap Aggregating) classifier.

    Parameters
    ----------
    base_estimator_class : class
        The base estimator class (e.g., DecisionTreeClassifier).
    n_estimators : int, default=10
        Number of base estimators.
    random_state : int, default=42
        Random state for reproducibility.
    base_estimator_kwargs : dict or None, default=None
        Additional keyword arguments passed to each base estimator constructor.
    """

    def __init__(self, base_estimator_class, n_estimators=10, random_state=42, base_estimator_kwargs=None):
        self.base_estimator_class = base_estimator_class
        self.n_estimators = n_estimators
        self.random_state = random_state
        self.base_estimator_kwargs = base_estimator_kwargs or {}
        self.estimators_ = []

    def fit(self, X, y):
        """Fit the bagging ensemble.

        Parameters
        ----------
        X : ndarray of shape (n_samples, n_features)
        y : ndarray of shape (n_samples,)

        Returns
        -------
        self
        """
        # YOUR CODE HERE — use bootstrap_sample and majority_vote from earlier
        raise NotImplementedError("Implement fit")

    def predict(self, X):
        """Predict class labels using majority vote.

        Parameters
        ----------
        X : ndarray of shape (n_samples, n_features)

        Returns
        -------
        ndarray of shape (n_samples,)
        """
        # YOUR CODE HERE
        raise NotImplementedError("Implement predict")

In [ ]:
try:
    run_single_test(BaggingClassifier, check_bagging_classifier)
except Exception as e:
    print(f"⚠️ Skipped (not yet implemented or error): {e}")

### Visualizing Bagging vs Single Tree

Let's compare a single decision tree with a bagging ensemble.

In [ ]:
try:
    X_complex, y_complex = generate_demo_complex_data()
    single_tree = DecisionTreeClassifier(max_depth=5).fit(X_complex, y_complex)
    bag = BaggingClassifier(base_estimator_class=DecisionTreeClassifier, n_estimators=20, random_state=42).fit(
        X_complex, y_complex
    )
    plot_ensemble_vs_single(X_complex, y_complex, single_tree, bag, "Single Tree", "Bagging (20 trees)")
except Exception as e:
    print(f"⚠️ Skipped (not yet implemented or error): {e}")

---
## Section 4: Random Forest

A **Random Forest** extends bagging by adding **random feature subsampling** at each tree. Instead of considering all features for each split, each tree only sees a random subset of features.

**Key idea:** By using random feature subsets, trees become less correlated → the ensemble has lower variance.

Common choices for `max_features`:
- `"sqrt"`: $\sqrt{p}$ features (classification default)
- `"log2"`: $\log_2(p)$ features
- `int`: exact number

### Exercise 4.1: Feature Subsampling

Implement a function that selects a random subset of feature indices.

**Parameters:**
- `n_features`: total number of features
- `max_features`: `"sqrt"`, `"log2"`, or an integer
- `random_state`: for reproducibility

In [ ]:
def subsample_features(n_features, max_features, random_state=None):
    """Select a random subset of feature indices.

    Parameters
    ----------
    n_features : int
        Total number of features.
    max_features : str or int
        "sqrt", "log2", or an integer.
    random_state : int or None

    Returns
    -------
    ndarray of int : selected feature indices (sorted)
    """
    # YOUR CODE HERE
    raise NotImplementedError("Implement subsample_features")

In [ ]:
try:
    idx = subsample_features(10, "sqrt", random_state=42)
    assert len(idx) == 3, f"sqrt(10) should give 3 features, got {len(idx)}"
    assert np.all(idx >= 0) and np.all(idx < 10), "Indices should be in [0, 10)"

    idx2 = subsample_features(10, "sqrt", random_state=0)
    assert not np.array_equal(idx, idx2), "Different seeds should give different subsets"

    idx3 = subsample_features(10, 5, random_state=42)
    assert len(idx3) == 5, f"max_features=5 should give 5 features, got {len(idx3)}"
    print("All subsample_features tests passed! ✓")
except Exception as e:
    print(f"⚠️ Skipped (not yet implemented or error): {e}")

### Exercise 4.2: RandomForestClassifier

Build a Random Forest that combines bagging with feature subsampling.

**Key differences from BaggingClassifier:**
- Each tree is trained on a **bootstrap sample** AND a **random subset of features**
- At prediction time, each tree only uses its assigned features
- Store `self.feature_indices_` (list of arrays) to track which features each tree uses

In [ ]:
class RandomForestClassifier:
    """Random Forest classifier.

    Parameters
    ----------
    n_estimators : int, default=10
        Number of trees.
    max_depth : int, default=10
        Maximum depth of each tree.
    max_features : str or int, default="sqrt"
        Number of features to consider for each tree.
    random_state : int, default=42
        Random state for reproducibility.
    """

    def __init__(self, n_estimators=10, max_depth=10, max_features="sqrt", random_state=42):
        self.n_estimators = n_estimators
        self.max_depth = max_depth
        self.max_features = max_features
        self.random_state = random_state
        self.estimators_ = []
        self.feature_indices_ = []

    def fit(self, X, y):
        """Fit the random forest.

        Parameters
        ----------
        X : ndarray of shape (n_samples, n_features)
        y : ndarray of shape (n_samples,)

        Returns
        -------
        self
        """
        # YOUR CODE HERE — use bootstrap_sample and subsample_features from earlier
        raise NotImplementedError("Implement fit")

    def predict(self, X):
        """Predict class labels using majority vote.

        Parameters
        ----------
        X : ndarray of shape (n_samples, n_features)

        Returns
        -------
        ndarray of shape (n_samples,)
        """
        # YOUR CODE HERE
        raise NotImplementedError("Implement predict")

In [ ]:
try:
    run_single_test(RandomForestClassifier, check_random_forest)
except Exception as e:
    print(f"⚠️ Skipped (not yet implemented or error): {e}")

### Visualizing Random Forest vs Bagging

Let's compare Random Forest with plain Bagging on the complex dataset.

In [ ]:
try:
    X_complex, y_complex = generate_demo_complex_data()
    bag = BaggingClassifier(base_estimator_class=DecisionTreeClassifier, n_estimators=20, random_state=42).fit(
        X_complex, y_complex
    )
    rf = RandomForestClassifier(n_estimators=20, max_depth=10, max_features=2, random_state=42).fit(
        X_complex, y_complex
    )
    plot_ensemble_vs_single(X_complex, y_complex, bag, rf, "Bagging (20 trees)", "Random Forest (20 trees)")
except Exception as e:
    print(f"⚠️ Skipped (not yet implemented or error): {e}")

---
## Section 5: Gradient Boosting

**Gradient Boosting** builds an additive model by sequentially fitting weak learners (small trees) to the **negative gradient** of the loss function.

For binary classification with log loss:

$$L(y, F) = -\frac{1}{n}\sum_i \left[ y_i \log(p_i) + (1 - y_i) \log(1 - p_i) \right]$$

where $p_i = \sigma(F_i)$ is the sigmoid of the raw prediction.

The negative gradient (pseudo-residual) is simply: $r_i = y_i - p_i$

**Algorithm:**
1. Initialize $F_0$ with the log-odds of the positive class
2. For $m = 1, \ldots, M$:
   - Compute residuals $r_i = y_i - \sigma(F_{m-1}(x_i))$
   - Fit a regression tree (stump) $h_m$ to the residuals
   - Update $F_m = F_{m-1} + \eta \cdot h_m$
3. Predict probabilities: $p = \sigma(F_M)$

Let's build this step by step!

### Exercise 5.1: Decision Stump (Regression)

A **decision stump** is a depth-1 decision tree for **regression** (not classification!). This is the weak learner used in gradient boosting.

The stump:
- Finds the best single split that minimizes mean squared error (MSE)
- Each leaf predicts the **mean** of its training targets

**Methods:**
- `fit(X, y)` — find the best split (feature and threshold) minimizing MSE
- `predict(X)` — return the mean of the appropriate leaf for each sample

In [ ]:
class DecisionStump:
    """Decision stump (depth-1 regression tree).

    Used as the weak learner in gradient boosting.

    Parameters
    ----------
    random_state : int or None
        Not used, included for API compatibility.
    """

    def __init__(self, random_state=None):
        self.random_state = random_state
        self.feature_ = None
        self.threshold_ = None
        self.left_value_ = None
        self.right_value_ = None
        self.default_value_ = None

    def fit(self, X, y):
        """Fit the stump by finding the best single split (minimizing MSE).

        Parameters
        ----------
        X : ndarray of shape (n_samples, n_features)
        y : ndarray of shape (n_samples,)

        Returns
        -------
        self
        """
        # YOUR CODE HERE
        raise NotImplementedError("Implement fit")

    def predict(self, X):
        """Predict regression values.

        Parameters
        ----------
        X : ndarray of shape (n_samples, n_features)

        Returns
        -------
        ndarray of shape (n_samples,)
        """
        # YOUR CODE HERE
        raise NotImplementedError("Implement predict")

In [ ]:
try:
    X_reg = np.array([[1], [2], [3], [4], [5]])
    y_reg = np.array([1.0, 1.0, 5.0, 5.0, 5.0])
    stump = DecisionStump().fit(X_reg, y_reg)
    preds = stump.predict(X_reg)
    assert preds.shape == (5,), f"Expected shape (5,), got {preds.shape}"
    assert abs(preds[0] - 1.0) < 0.5, f"Left prediction should be ~1.0, got {preds[0]}"
    assert abs(preds[-1] - 5.0) < 0.5, f"Right prediction should be ~5.0, got {preds[-1]}"
    print("All DecisionStump tests passed! ✓")
except Exception as e:
    print(f"⚠️ Skipped (not yet implemented or error): {e}")

### Exercise 5.2: Negative Gradient (Residuals)

For binary classification with log loss, the negative gradient (pseudo-residual) is:

$$r_i = y_i - p_i$$

where $p_i = \sigma(F_i)$ is the predicted probability.

In [ ]:
def negative_gradient(y, probs):
    """Compute negative gradient (pseudo-residuals) for log loss.

    Parameters
    ----------
    y : ndarray of shape (n_samples,) — true labels (0 or 1)
    probs : ndarray of shape (n_samples,) — predicted probabilities

    Returns
    -------
    ndarray of shape (n_samples,) — residuals
    """
    # YOUR CODE HERE
    raise NotImplementedError("Implement negative_gradient")

In [ ]:
try:
    y_test = np.array([1, 0, 1, 0])
    p_test = np.array([0.9, 0.1, 0.3, 0.8])
    residuals = negative_gradient(y_test, p_test)
    expected = np.array([0.1, -0.1, 0.7, -0.8])
    assert np.allclose(residuals, expected), f"Expected {expected}, got {residuals}"
    print("All negative_gradient tests passed! ✓")
except Exception as e:
    print(f"⚠️ Skipped (not yet implemented or error): {e}")

### Exercise 5.3: Update Predictions

In gradient boosting, we update the raw predictions by adding the weak learner's contribution scaled by the learning rate:

$$F_m(x) = F_{m-1}(x) + \eta \cdot h_m(x)$$

In [ ]:
def update_predictions(raw_predictions, learning_rate, stump_predictions):
    """Update raw predictions with a boosting step.

    Parameters
    ----------
    raw_predictions : ndarray of shape (n_samples,)
    learning_rate : float
    stump_predictions : ndarray of shape (n_samples,)

    Returns
    -------
    ndarray of shape (n_samples,) — updated raw predictions
    """
    # YOUR CODE HERE
    raise NotImplementedError("Implement update_predictions")

In [ ]:
try:
    raw = np.array([0.0, 1.0, -1.0])
    lr = 0.1
    stump_pred = np.array([1.0, -1.0, 2.0])
    updated = update_predictions(raw, lr, stump_pred)
    expected = np.array([0.1, 0.9, -0.8])
    assert np.allclose(updated, expected), f"Expected {expected}, got {updated}"
    print("All update_predictions tests passed! ✓")
except Exception as e:
    print(f"⚠️ Skipped (not yet implemented or error): {e}")

### Exercise 5.4: GradientBoostingClassifier

Combine the building blocks into a full `GradientBoostingClassifier` for binary classification.

**Algorithm:**
1. Initialize raw predictions $F_0 = \log\left(\frac{p}{1-p}\right)$ where $p$ is the proportion of positive class
2. For each boosting round:
   - Compute probabilities: $p_i = \sigma(F(x_i))$
   - Compute residuals: $r_i = y_i - p_i$
   - Fit a `DecisionStump` to $(X, r)$
   - Update: $F \leftarrow F + \eta \cdot h(X)$
   - Compute and store log loss

**Sigmoid:** Use `np.clip(z, -500, 500)` before computing `1 / (1 + np.exp(-z))` for numerical stability.

In [ ]:
class GradientBoostingClassifier:
    """Gradient Boosting classifier for binary classification.

    Parameters
    ----------
    n_estimators : int, default=100
        Number of boosting rounds.
    learning_rate : float, default=0.1
        Shrinkage factor.
    random_state : int, default=42
        Random state for reproducibility.
    """

    def __init__(self, n_estimators=100, learning_rate=0.1, random_state=42):
        self.n_estimators = n_estimators
        self.learning_rate = learning_rate
        self.random_state = random_state
        self.estimators_ = []
        self.initial_prediction_ = None
        self.losses_ = []

    def _sigmoid(self, z):
        """Numerically stable sigmoid."""
        z = np.clip(z, -500, 500)
        return 1.0 / (1.0 + np.exp(-z))

    def _log_loss(self, y, probs):
        """Compute binary log loss."""
        probs = np.clip(probs, 1e-15, 1 - 1e-15)
        return -np.mean(y * np.log(probs) + (1 - y) * np.log(1 - probs))

    def fit(self, X, y):
        """Fit the gradient boosting model.

        Parameters
        ----------
        X : ndarray of shape (n_samples, n_features)
        y : ndarray of shape (n_samples,) — binary labels (0 or 1)

        Returns
        -------
        self
        """
        # YOUR CODE HERE — use negative_gradient, DecisionStump, and update_predictions
        raise NotImplementedError("Implement fit")

    def predict_proba(self, X):
        """Predict class probabilities.

        Parameters
        ----------
        X : ndarray of shape (n_samples, n_features)

        Returns
        -------
        ndarray of shape (n_samples,) — probability of class 1
        """
        # YOUR CODE HERE
        raise NotImplementedError("Implement predict_proba")

    def predict(self, X):
        """Predict class labels.

        Parameters
        ----------
        X : ndarray of shape (n_samples, n_features)

        Returns
        -------
        ndarray of shape (n_samples,) — predicted labels (0 or 1)
        """
        # YOUR CODE HERE
        raise NotImplementedError("Implement predict")

In [ ]:
try:
    run_single_test(GradientBoostingClassifier, check_gradient_boosting)
except Exception as e:
    print(f"⚠️ Skipped (not yet implemented or error): {e}")

### Visualizing Gradient Boosting

Let's see the loss curve and the decision boundary.

In [ ]:
try:
    X_demo_bin, y_demo_bin = generate_demo_binary_data()
    gbm = GradientBoostingClassifier(n_estimators=100, learning_rate=0.1, random_state=42).fit(X_demo_bin, y_demo_bin)
    plot_boosting_loss_curve(gbm.losses_)
except Exception as e:
    print(f"⚠️ Skipped (not yet implemented or error): {e}")

In [ ]:
try:
    plot_decision_boundary_2d(X_demo_bin, y_demo_bin, gbm, "Gradient Boosting Decision Boundary")
except Exception as e:
    print(f"⚠️ Skipped (not yet implemented or error): {e}")

### Comparing All Methods

Let's compare all four methods on the complex dataset.

In [ ]:
try:
    X_complex, y_complex = generate_demo_complex_data()
    split = len(X_complex) // 2
    X_train, y_train = X_complex[:split], y_complex[:split]
    X_test, y_test = X_complex[split:], y_complex[split:]

    models = {
        "Decision Tree": DecisionTreeClassifier(max_depth=10).fit(X_train, y_train),
        "Bagging (20 trees)": BaggingClassifier(
            DecisionTreeClassifier,
            n_estimators=20,
            random_state=42,
        ).fit(X_train, y_train),
        "Random Forest (20)": RandomForestClassifier(
            n_estimators=20, max_depth=10, max_features=2, random_state=42
        ).fit(X_train, y_train),
        "Gradient Boosting": GradientBoostingClassifier(n_estimators=200, learning_rate=0.1, random_state=42).fit(
            X_train, y_train
        ),
    }

    fig, axes = plt.subplots(2, 2, figsize=(14, 12))
    x_min, x_max = X_complex[:, 0].min() - 0.5, X_complex[:, 0].max() + 0.5
    y_min, y_max = X_complex[:, 1].min() - 0.5, X_complex[:, 1].max() + 0.5
    xx, yy = np.meshgrid(np.linspace(x_min, x_max, 150), np.linspace(y_min, y_max, 150))
    grid = np.column_stack([xx.ravel(), yy.ravel()])

    for ax, (name, model) in zip(axes.ravel(), models.items()):
        Z = model.predict(grid).reshape(xx.shape)
        train_acc = np.mean(model.predict(X_train) == y_train)
        test_acc = np.mean(model.predict(X_test) == y_test)
        ax.contourf(xx, yy, Z, alpha=0.3, cmap="RdYlBu")
        ax.scatter(X_test[:, 0], X_test[:, 1], c=y_test, cmap="RdYlBu", edgecolors="k", s=20)
        ax.set_title(f"{name}\ntrain={train_acc:.1%}  test={test_acc:.1%}")
        ax.grid(True, alpha=0.3)

    plt.suptitle("Comparison of All Methods on Complex Data", fontsize=14)
    plt.tight_layout()
    plt.show()
except Exception as e:
    print(f"⚠️ Skipped (not yet implemented or error): {e}")

---
## Section 6: Summary & Key Takeaways

| Method | Type | Key Idea | Pros | Cons |
|--------|------|----------|------|------|
| Decision Tree | Single model | Greedy recursive splitting | Interpretable, handles non-linearity | High variance, overfits easily |
| Bagging | Ensemble (parallel) | Bootstrap + majority vote | Reduces variance | Many trees needed, no feature selection |
| Random Forest | Ensemble (parallel) | Bagging + random features | Less correlated trees, robust | Harder to interpret than single tree |
| Gradient Boosting | Ensemble (sequential) | Fit residuals iteratively | Very accurate, handles complex patterns | Can overfit, sequential (slow) |

**When to use what:**
- **Decision Tree**: when interpretability matters and data is small
- **Random Forest**: general-purpose, hard to go wrong
- **Gradient Boosting**: when you need maximum predictive accuracy
- **Bagging**: when you want to reduce variance of any base model

---
## Section 7: Run All Tests

In [ ]:
run_tests(
    DecisionTreeClass=DecisionTreeClassifier,
    BaggingClass=BaggingClassifier,
    RandomForestClass=RandomForestClassifier,
    GradientBoostingClass=GradientBoostingClassifier,
)